# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, overview, and explore a mass spectrometry dataset of extracellular vesicles from human mast cells using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Title: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id` for referencing via `mlcroissant`.

In [ ]:
# Get an overview of record sets, fields, and columns by their `@id`

def print_record_sets_and_fields(ds):
    print("Available Record Sets and Their Fields (by @id):\n")
    for rs in ds.record_sets:
        print(f"Record Set: {rs.id}")
        # List available fields for this record set
        if hasattr(rs, 'fields'):
            for field in rs.fields:
                print(f"  Field: {field.id} (Name: {getattr(field, 'name', 'N/A')})")
        else:
            print("  No fields found.")
        print()

print_record_sets_and_fields(dataset)

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrame for analysis. Use the record set and field `@id` values from the overview.

In [ ]:
# Collect all record set @ids
record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}

# Load each record set into a pandas DataFrame
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for {record_set_id} with shape: {dataframes[record_set_id].shape}")

# If at least one record set is present, print its columns and first rows
if record_sets:
    first_rs = record_sets[0]
    print(f"\nColumns for record set {first_rs}:\n", dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Perform basic processing steps using field and column `@id` references. Filter, normalize, or group as appropriate.

In [ ]:
# Select a record set and numeric field for demonstration
# Replace these with concrete `@id` values from the overview section as appropriate
if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id].copy()

    # Try to automatically select a numeric column
    numeric_fields = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")

        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != object else 0
        # Filter based on the threshold, handle non-numeric gracefully
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (Total: {filtered_df.shape[0]} records):\n")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records (showing original and normalized values):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a non-numeric field if present
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped data by {group_field} (showing mean {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric fields detected in the selected record set.")
else:
    print("No record sets found in dataset.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset (e.g., histograms, scatter plots, group means).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_fields:
    # Histogram of numeric field
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If a grouping field is present, show group means
    if group_field is not None:
        group_means = df.groupby(group_field)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field_id, data=group_means)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we have demonstrated how to access a mass spectrometry dataset using `mlcroissant`, explored its structure (using `@id` references as per Croissant), extracted and previewed tabular data from record sets, and performed basic exploratory analysis including filtering, normalization, grouping, and visualization. 

This workflow can be easily adapted to other FAIR-compliant datasets described with Croissant schemas for reproducible data loading and analysis.